### Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching web, running code, Tools are pairings of:

    1. A schema,including the name of the tool, a descroption, and/or argument definition (often a JSON schema)
    2. A function or coroutine to execute.

In [3]:
import os
from dotenv import load_dotenv

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
load_dotenv()
from langchain.chat_models import init_chat_model
model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Why do parrtos talk")
response

AIMessage(content='<think>\nOkay, the user is asking "Why do parrtos talk". First, I need to figure out what "parrtos" refers to. It might be a typo or a misspelling of a word. Let me consider possible corrections. Common words that come to mind are "parties" (like political parties), "parents", "parties" as in social gatherings, or maybe a specific term. The user might have meant "parties" instead of "parrtos".\n\nAssuming it\'s a typo for "parties", the question becomes "Why do parties talk". Now, in the context of political parties, they might talk for various reasons like forming alliances, negotiating policies, or discussing governance. If it\'s about social parties, people might talk to socialize, network, or share information. Another possibility is if "parrtos" refers to a specific group or term in a certain field, but without more context, it\'s hard to say.\n\nI should also consider if "parrtos" is a name of a specific group or entity. Maybe it\'s a local term or an organizat

In [5]:
# Creating a tool
from langchain.tools import tool

@tool
def get_weather(location: str)->str:
    """Get the weather at a location."""
    return f"it is sunny in {location}"

mode_with_tools = model.bind_tools([get_weather])

In [6]:
response = mode_with_tools.invoke("What is weather in houston?")
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args:{tool_call["args"]}") 

Tool: get_weather
Args:{'location': 'Houston'}


In [7]:
### Tool Execution loop

# Step-1 Model generates tool calls
messages =[{"role":"user", "content":"What is weather in new jersey"}]
ai_msg = mode_with_tools.invoke(messages)
messages.append(ai_msg)

# Step-2 Execute the tools and collect results
for  tool_call in ai_msg.tool_calls:
    # Execute the tool with generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step-3 Pass results back to model for final response
final_response = mode_with_tools.invoke(messages)
print(final_response.text)

The weather in New Jersey is sunny. Let me know if you need more details! ☀️


In [8]:
messages

[{'role': 'user', 'content': 'What is weather in new jersey'},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in New Jersey. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. Since the user specified "new jersey", I need to call that function with the location set to "New Jersey". I should make sure the arguments are correctly formatted as JSON. Let me structure the tool call accordingly.\n', 'tool_calls': [{'id': 'f0q7pv1a4', 'function': {'arguments': '{"location":"New Jersey"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 101, 'prompt_tokens': 152, 'total_tokens': 253, 'completion_time': 0.145786474, 'completion_tokens_details': {'reasoning_tokens': 76}, 'prompt_time': 0.007428982, 'prompt_tokens_details': None, 'queue_time': 0.452837294, 'total_time': 0.153215456}, 'model_name': 'qwen/qwen3-32b', 'system_fingerpr